# Notebook 01 — Simulation EDA

This notebook explores the **Markov-switching AR(1)** data-generating process and
documents the Lucas-critique structural break.

**Topics covered**
- DGP parameters and regime properties
- Simulated series visualisation with regime shading
- Regime duration and transition matrix analysis
- Lucas shift: how the DGP changes post-break
- Feature distributions by regime

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Add src to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from simulation import MarkovSwitchingDGP, RegimeParams, LucasShift, apply_lucas_shift, simulate_pre_post_break
from simulation.lucas_shift import MILD_SHIFT, SEVERE_SHIFT, concatenate_periods
from evaluation.visualization import (
    plot_simulated_series,
    plot_regime_transition_heatmap,
    save_figure,
)

sns.set_theme(style='whitegrid', context='notebook')
FIGURES_DIR = PROJECT_ROOT / 'analyses' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 1. DGP Parameters

In [ ]:
dgp = MarkovSwitchingDGP(seed=42)
print(dgp)
print()
print('Transition matrix:')
print(dgp.transition)
print()
print('Stationary distribution:', dgp.stationary_distribution.round(4))

## 2. Simulated Series

In [ ]:
df = dgp.simulate(n_obs=600)
print(f'Simulated {len(df)} observations after dropping NaN warm-up rows.')
print(df.head())

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
plot_simulated_series(df, title='Simulated Markov-Switching AR(1): Pre-break DGP', ax=ax)
save_figure(fig, '01_simulated_series')
plt.show()

## 3. Regime Statistics

In [ ]:
print('Regime counts and proportions:')
print(df['regime_label'].value_counts())
print()
print('Descriptive statistics by regime:')
print(df.groupby('regime_label')['y'].agg(['mean', 'std', 'min', 'max']).round(4))

In [ ]:
# Regime duration analysis
regime_seq = df['regime'].to_numpy()
durations = {0: [], 1: []}
count = 1
for i in range(1, len(regime_seq)):
    if regime_seq[i] == regime_seq[i - 1]:
        count += 1
    else:
        durations[regime_seq[i - 1]].append(count)
        count = 1
durations[regime_seq[-1]].append(count)

print('Regime duration summary:')
for reg, label in enumerate(['recession', 'expansion']):
    d = durations[reg]
    print(f'  {label}: mean={np.mean(d):.1f}, median={np.median(d):.1f}, max={max(d)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for reg, (label, color) in enumerate(zip(['recession', 'expansion'], ['#d62728', '#2ca02c'])):
    axes[reg].hist(durations[reg], bins=15, color=color, edgecolor='white', alpha=0.85)
    axes[reg].set_title(f'Duration distribution: {label}')
    axes[reg].set_xlabel('Duration (periods)')
    axes[reg].set_ylabel('Count')
plt.tight_layout()
save_figure(fig, '01_regime_durations')
plt.show()

## 4. Transition Matrix Heatmap

In [ ]:
ax = plot_regime_transition_heatmap(
    dgp.transition,
    regime_labels=['Recession', 'Expansion'],
    title='Pre-break Regime Transition Matrix',
)
save_figure(ax.get_figure(), '01_transition_matrix_pre')
plt.show()

## 5. Lucas Shift: Pre vs Post-Break DGP

In [ ]:
# Show the parameter changes
post_dgp = apply_lucas_shift(dgp, MILD_SHIFT)

print('Parameter comparison: Pre-break vs Post-break (Mild shift)')
print(f'{"Regime":<15} {"param":<10} {"pre":<10} {"post":<10} {"delta":<10}')
print('-' * 55)
for i, (pre_r, post_r) in enumerate(zip(dgp.regimes, post_dgp.regimes)):
    for param in ['mu', 'phi', 'sigma']:
        pre_val = getattr(pre_r, param)
        post_val = getattr(post_r, param)
        print(f'{pre_r.label:<15} {param:<10} {pre_val:<10.3f} {post_val:<10.3f} {post_val - pre_val:<10.3f}')
print()
print('Pre-break transition matrix:')
print(dgp.transition)
print('Post-break transition matrix:')
print(post_dgp.transition)

In [ ]:
df_pre, df_post, post_dgp_used = simulate_pre_post_break(
    dgp, MILD_SHIFT, n_pre=400, n_post=200
)
df_full = concatenate_periods(df_pre, df_post)

fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=False)
plot_simulated_series(df_pre, title='Pre-break period (original DGP)', ax=axes[0])
plot_simulated_series(df_post, title='Post-break period (Lucas-shifted DGP)', ax=axes[1])
plt.tight_layout()
save_figure(fig, '01_pre_vs_post_break')
plt.show()

## 6. Feature Distributions by Regime

In [ ]:
feature_cols = ['y', 'y_lag1', 'roll_mean_5', 'roll_std_5', 'exog_0']
fig, axes = plt.subplots(1, len(feature_cols), figsize=(16, 4))

for ax, col in zip(axes, feature_cols):
    for reg, (label, color) in enumerate(zip(['recession', 'expansion'], ['#d62728', '#2ca02c'])):
        data = df_pre[df_pre['regime'] == reg][col]
        ax.hist(data, bins=30, alpha=0.55, color=color, label=label, density=True)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

fig.suptitle('Feature distributions by regime (pre-break)', fontsize=12)
plt.tight_layout()
save_figure(fig, '01_feature_distributions')
plt.show()

## 7. Save Data for Subsequent Notebooks

In [ ]:
from pathlib import Path
data_dir = PROJECT_ROOT / 'data' / 'simulated'
data_dir.mkdir(parents=True, exist_ok=True)

df_pre.to_parquet(data_dir / 'pre_break.parquet', index=False)
df_post.to_parquet(data_dir / 'post_break.parquet', index=False)
df_full.to_parquet(data_dir / 'full_series.parquet', index=False)

print('Saved:')
for f in sorted(data_dir.glob('*.parquet')):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} KB)')